> **Current notebook role — 2026-11-12.** This notebook retains the upstream stage code used by the manuscript implementation. Selected figure production uses `run.py` in `code_release/iCCA-molecular-subtyping` or the public source repository. See `docs/NOTEBOOK_STATUS.md` in that code repository for stage ownership. Earlier execution outputs are preserved in the archived notebook snapshot.

# Step 1 — Cross-study molecular subtype discovery and robustness

## tl;dr

Seven RNA-sequencing cohorts contribute 586 discovery tumors. Study identity dominates the uncorrected expression structure, whereas an exact memory-only reconstruction of the locked `limma::removeBatchEffect` result matches the stored corrected matrix (maximum absolute difference below 1e-13). Candidate structures are evaluated separately by consensus clustering, NMF, KMeans, and hierarchical clustering. Consensus and hierarchical results are degenerate and excluded. NMF and KMeans independently support an operational k=3 solution; their 497 concordant samples define C1/C2/C3 and 89 disagreements define three explicit pairwise intermediate states. Exact 50-run NMF and KMeans refits reproduce all 586 locked assignments.

## Context & Methods

This is the discovery notebook, not a biological-characterization notebook. It proceeds from cross-study integration and batch handling to feature selection, method-specific unsupervised clustering, method disposition, state definition, and robustness. DEG, pathway, TF, TME, and subtype biological interpretation are owned by `2-transcriptomic-characterization.ipynb`.

### Key assumptions and boundaries

- Step0 provides per-study processed log2-scale expression matrices; Step1 owns cross-study correction for discovery.
- The historical locked correction used Study as the batch factor and supplied no biological design. This is reproduced for audit without overwriting either matrix.
- The top 2,000 genes by variance in the corrected discovery matrix are part of the locked clustering interface.
- Consensus and hierarchical outputs are evaluated but are not counted as supporting votes when they produce degenerate partitions.
- NMF and KMeans cluster IDs are permutation-aligned only after their k=3 solutions have been assessed.
- PCA is a diagnostic visualization, not evidence for choosing k.

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(readxl)
})

set.seed(123)
root <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
data_dir <- file.path(root, "1-Clustering", "data")
table_dir <- file.path(root, "1-Clustering", "publication", "tables")
figure_dir <- file.path(root, "1-Clustering", "publication", "figures")
study_palette <- fread(file.path(root, "0-Data", "qc", "study_color_dictionary.csv"))
subtype_palette <- fread(file.path(root, "docs", "subtype_color_dictionary.csv"))
stopifnot(
  !anyDuplicated(study_palette$Study),
  !anyDuplicated(subtype_palette$FinalSubtype),
  all(grepl("^#[0-9A-Fa-f]{6}$", study_palette$ColorHex)),
  all(grepl("^#[0-9A-Fa-f]{6}$", subtype_palette$ColorHex))
)

refresh_discovery <- identical(Sys.getenv("SCLUCID_STEP1_DISCOVERY_REFRESH", "0"), "1")
cat("Execution mode:", ifelse(refresh_discovery, "full discovery refresh", "locked-interface validation"), "
")

## Data

### 1. Load the pre-correction, corrected, and sample-level discovery interfaces

In [ ]:
load_object <- function(path, object_name) {
  env <- new.env(parent = emptyenv())
  load(path, envir = env)
  stopifnot(exists(object_name, envir = env, inherits = FALSE))
  get(object_name, envir = env, inherits = FALSE)
}

expression_before <- load_object(file.path(data_dir, "combined_expr_before_correction.rds"), "combined_expr")
expression_after <- load_object(file.path(data_dir, "combined_expr_after_correction.rds"), "corrected_expr_limma")
sample_info <- as.data.table(read_excel(file.path(data_dir, "sample_info_with_clusters.xlsx")))
top_variable <- fread(file.path(data_dir, "top_variable_genes.csv"))
sample_info[, Sample := as.character(Sample)]

stopifnot(
  identical(dim(expression_before), c(10531L, 586L)),
  identical(dim(expression_after), c(10531L, 586L)),
  identical(rownames(expression_before), rownames(expression_after)),
  identical(colnames(expression_before), colnames(expression_after)),
  identical(sort(sample_info$Sample), sort(colnames(expression_after))),
  nrow(top_variable) == 2000L
)

data.table(
  Interface = c("Before study correction", "Locked limma correction", "Locked variable-gene set"),
  Genes = c(nrow(expression_before), nrow(expression_after), nrow(top_variable)),
  Samples = c(ncol(expression_before), ncol(expression_after), NA_integer_)
)

### 2. Discovery cohort composition

In [ ]:
cohort_summary <- sample_info[, .N, by = Study][order(match(Study, study_palette$Study))]
stopifnot(sum(cohort_summary$N) == 586L, nrow(cohort_summary) == 7L)
cohort_summary

## Results

### 1. Diagnose and reproduce study correction

In [ ]:
batch_script <- file.path(root, "1-Clustering", "scripts", "run_step1_batch_and_feature_audit.R")
stopifnot(file.exists(batch_script))
sys.source(batch_script, envir = new.env(parent = globalenv()))

In [ ]:
batch_contract <- fread(file.path(table_dir, "step1_batch_correction_contract.csv"))
batch_pc <- fread(file.path(table_dir, "step1_batch_pc_study_association.csv"))
batch_cluster <- fread(file.path(table_dir, "step1_batch_handling_cluster_sensitivity.csv"))

batch_contract
batch_pc[PCIndex <= 3L, .(
  CorrectionState, PC, VarianceExplainedPercent,
  StudyEtaSquared, PValue
)]
batch_cluster

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_batch_pca_before_after.png"))
IRdisplay::display_png(file = file.path(figure_dir, "step1_batch_pc_study_association.png"))

The uncorrected k=3 solutions are strongly study-associated and do not reproduce the locked biological partition. The correction is therefore necessary for this cross-study interface. Because the historical model supplied no biological design, a near-zero post-correction Study association is an algebraic consequence of removing Study means, not sufficient evidence against overcorrection. The downstream safeguards are explicit multi-study representation, study-adjusted biological models, held-study-out clustering, and locked-versus-refit sensitivity.

### 2. Select the locked top-2,000 variance genes

In [ ]:
feature_summary <- fread(file.path(table_dir, "step1_feature_selection_summary.csv"))
feature_summary
head(top_variable, 8L)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_feature_selection_variance.png"))

### 3. Evaluate candidate unsupervised structures separately by method

In [ ]:
if (refresh_discovery) {
  refresh_scripts <- file.path(root, "1-Clustering", "scripts", c(
    "run_step1_clustering_stability.R",
    "run_step1_operational_structure_sensitivity.R",
    "run_step1_locked_vs_refit.R"
  ))
  stopifnot(all(file.exists(refresh_scripts)))
  for (script in refresh_scripts) {
    cat("Running", basename(script), "
")
    sys.source(script, envir = new.env(parent = globalenv()))
  }
} else {
  cat("Saved method-specific source tables retained. Set SCLUCID_STEP1_DISCOVERY_REFRESH=1 to recompute consensus, NMF, KMeans, and exact refits.
")
}

sys.source(
  file.path(root, "1-Clustering", "scripts", "plot_step1_discovery_sequence.R"),
  envir = new.env(parent = globalenv())
)

In [ ]:
method_parameters <- fread(file.path(table_dir, "step1_clustering_methods_parameters.csv"))
method_parameters

#### 3.1 ConsensusClusterPlus sensitivity

In [ ]:
consensus_metrics <- fread(file.path(table_dir, "step1_standard_consensus_pac_cdf.csv"))
consensus_sizes <- fread(file.path(table_dir, "step1_standard_consensus_cluster_sizes.csv"))
consensus_metrics
consensus_sizes[K == 3][order(-N)]

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_standard_consensus_stability.png"))

The matched standard consensus analysis is not evidence for three balanced states: at k=3, approximately 99.5% of samples occupy one cluster. This method is retained as a documented failed sensitivity analysis.

#### 3.2 NMF rank assessment and exact k=3 fit

In [ ]:
nmf_rank <- fread(file.path(table_dir, "step1_nmf_rank_sensitivity_10run.csv"))
nmf_rank[, .(
  K, Runs, Cophenetic, Dispersion, StandardPAC_0.1_0.9,
  MeanSilhouetteEuclidean, SmallestCluster, LargestCluster,
  ARIvsHistoricalNMF
)]

NMF retains k=3 as the operational rank because cophenetic stability peaks at k=3, PAC is low, the solution is non-degenerate, and the independent exact 50-run refit reproduces the locked NMF labels. The historical notebook's manual fallback to k=3 is therefore replaced by an explicit evidence record.

#### 3.3 KMeans candidate k and seed stability

In [ ]:
kmeans_rank <- fread(file.path(table_dir, "step1_kmeans_rank_sensitivity.csv"))
kmeans_seed <- fread(file.path(table_dir, "step1_kmeans_k3_seed_sensitivity.csv"))
kmeans_rank[, .(
  K, NStart, BetweenSSFraction, MeanSilhouette,
  SmallestCluster, LargestCluster, ARIvsHistoricalKMeans
)]
kmeans_seed[, .(
  Seeds = .N,
  MinimumARI = min(ARIvsHistoricalKMeans),
  MedianARI = median(ARIvsHistoricalKMeans)
)]

#### 3.4 Hierarchical clustering and historical ensemble disposition

In [ ]:
method_disposition <- fread(file.path(table_dir, "step1_method_disposition.csv"))
method_disposition

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_method_selection_summary.png"))

### 4. Compare methods before defining states

In [ ]:
method_agreement <- fread(file.path(table_dir, "step1_stored_method_pairwise_agreement.csv"))
method_agreement

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_cross_method_agreement.png"))

Only NMF and KMeans are retained as the operational pair. Their ARI is approximately 0.619: this is substantial but incomplete agreement, so the final representation must preserve disagreement rather than describe all 586 tumors as confidently belonging to one of three hard classes. Consensus/HC degeneracy and the historical four-method ensemble are not used as votes for C1/C2/C3.

### 5. Define three principal and three intermediate states

In [ ]:
mapping <- fread(file.path(table_dir, "step1_nmf_kmeans_state_definition_full.csv"))
state_counts <- mapping[, .(N = sum(N)), by = .(FinalSubtype = Ensemble)][FinalSubtype != "None"]
state_counts[, DisplayOrder := match(FinalSubtype, c("C1", "C2", "C3", "C1_C2", "C1_C3", "C2_C3"))]
setorder(state_counts, DisplayOrder)
mapping
state_counts[, .(FinalSubtype, N)]
stopifnot(
  state_counts[FinalSubtype %in% c("C1", "C2", "C3"), sum(N)] == 497L,
  state_counts[!FinalSubtype %in% c("C1", "C2", "C3"), sum(N)] == 89L
)

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_nmf_kmeans_state_definition.png"))

### 6. Audit study representation after state definition

In [ ]:
study_distribution <- fread(file.path(table_dir, "step1_main_subtype_by_study.csv"))
study_test <- fread(file.path(table_dir, "step1_study_subtype_association.csv"))
study_test
dcast(study_distribution, Study ~ Ensemble, value.var = "N", fill = 0L)

### 7. Refit the locked operational models

In [ ]:
refit_summary <- fread(file.path(table_dir, "step1_locked_vs_refit_summary.csv"))
refit_samples <- fread(file.path(table_dir, "step1_locked_vs_refit_sample_comparison.csv"))
stopifnot(
  nrow(refit_samples) == 586L,
  all(refit_summary$ARI == 1),
  all(refit_summary$ChangedSamples == 0L)
)
refit_summary

In [ ]:
IRdisplay::display_png(file = file.path(figure_dir, "step1_locked_vs_refit_confusion.png"))
IRdisplay::display_png(file = file.path(figure_dir, "step1_refit_assignment_uncertainty.png"))

### 8. Seed, feature-number, rank, and leave-one-study-out robustness

In [ ]:
feature_sensitivity <- fread(file.path(table_dir, "step1_kmeans_feature_number_sensitivity.csv"))
kmeans_loso <- fread(file.path(table_dir, "step1_kmeans_leave_one_study_out.csv"))
operational_summary <- fread(file.path(table_dir, "step1_operational_structure_sensitivity_summary.csv"))
feature_sensitivity
kmeans_loso[, .(HeldOutStudy, HeldOutN, ARIAllVsHistoricalKMeans)]
operational_summary

### 9. Clustering-only release gate

In [ ]:
sys.source(
  file.path(root, "1-Clustering", "scripts", "finalize_step1_clustering_audit.R"),
  envir = new.env(parent = globalenv())
)
gate <- fread(file.path(table_dir, "step1_clustering_release_gate_status.csv"))
stopifnot(!any(gate$Blocking), !any(gate$Status == "FAIL"))
gate

## Takeaways

- Study correction is necessary for the combined discovery interface, is exactly reproducible, and is explicitly treated as a potential overcorrection sensitivity rather than as a neutral preprocessing detail.
- ConsensusClusterPlus and hierarchical clustering are shown but excluded because their partitions are degenerate.
- NMF and KMeans are the only retained operational methods; k=3 is supported by complementary rank, silhouette, seed, and exact-refit evidence.
- Three principal states are defined only where NMF and KMeans agree. Pairwise disagreements remain visible as three intermediate states.
- Locked labels are retained because exact refits reproduce all 586 assignments. Broader feature-number and leave-one-study-out sensitivity remain disclosed.
- Biological characterization proceeds in `2-transcriptomic-characterization.ipynb`; large-/small-duct phenotype exploration and classification proceed separately in `3-duct-type-analysis.ipynb`.